# MOSAIC-GNN — pan-cancer multi-omics graph neural network

**M**ulti-**O**mic **S**tructure-**A**daptive **I**nterpretable **C**ross-cancer GNN
on TCGA (BLCA, LIHC, SKCM, THCA).

Each patient is encoded as a *graph of genes* (mutation / CNA / expression /
methylation channels on nodes, PPI + pathway + co-alteration edges), pooled into
functional modules, and the resulting patient vector becomes a node in a
*learned patient-similarity graph*. A cohort-adversarial split of the latent
(`z_shared` vs `z_specific`) is what makes the representation transfer to a
held-out cancer type rather than memorising the cohort.

Heads: stage/grade/subtype classification, Cox risk, and a discrete-time hazard
curve — trained jointly.

### How to use this notebook
1. Run **Setup** and **Configuration**.
2. **Explore** the cohorts.
3. **Single fold** trains one model quickly and produces the figures.
4. **Full experiments** (CV, baselines, ablations, leave-one-cancer-out) are
   guarded by `RUN_FULL` — flip it on when you have GPU time.
5. **Interpretation** produces the gene / module tables and the paper figures.

## 1. Setup

In [ ]:
import os, sys, subprocess, glob, warnings, json, time
warnings.filterwarnings("ignore")

# --- locate the `mosaic` package -----------------------------------------
CANDIDATES = [
    ".", "..", "/kaggle/working", "/kaggle/working/mosaic-gnn",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/*/*"),
]
PKG_ROOT = next((os.path.abspath(p) for p in CANDIDATES
                 if os.path.isfile(os.path.join(p, "mosaic", "config.py"))), None)

if PKG_ROOT is None:
    # Fall back to a zip uploaded as a Kaggle dataset or into /kaggle/working.
    zips = glob.glob("/kaggle/input/**/mosaic*gnn*.zip", recursive=True) \
         + glob.glob("**/mosaic*gnn*.zip", recursive=True)
    if zips:
        import zipfile
        dest = "/kaggle/working/mosaic-gnn" if os.path.isdir("/kaggle/working") \
               else "./mosaic-gnn"
        zipfile.ZipFile(zips[0]).extractall(dest)
        PKG_ROOT = os.path.abspath(dest)
        print("extracted", zips[0], "->", dest)

assert PKG_ROOT, ("`mosaic` package not found. Upload mosaic-gnn.zip as a Kaggle "
                  "dataset (or put the mosaic/ folder next to this notebook).")
sys.path.insert(0, PKG_ROOT)
print("package root:", PKG_ROOT)

for pkg in ("torch", "sklearn", "scipy", "pandas", "matplotlib"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        {"sklearn": "scikit-learn"}.get(pkg, pkg)], check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 200, "display.max_columns", 80)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

# Optional extras -- present = extra baselines in the comparison table.
for extra in ("xgboost", "lightgbm", "catboost", "sksurv"):
    try:
        __import__(extra); print(f"  {extra}: available")
    except ImportError:
        print(f"  {extra}: not installed (that baseline is skipped)")

## 2. Configuration

Everything that defines a run lives in one `Config` object, saved alongside
every checkpoint — so a result is always reproducible from its JSON.

`RUN_FULL` controls the expensive sections (head-to-head cross-validation,
ablations, leave-one-cancer-out).

In [ ]:
from mosaic.config import Config
from mosaic.pipeline import Experiment

DATA_ROOT = "/kaggle/input/datasets/jannatuljerin/tcga-capstone"
OUT_DIR   = "/kaggle/working/runs" if os.path.isdir("/kaggle/working") else "./runs"

RUN_FULL        = True   # False -> only the single-fold walkthrough
OUTER_FOLDS     = 4      # head-to-head CV folds
REPEATS         = 1      # raise to >= 3 for the manuscript
ABLATION_FOLDS  = 2      # ablations are relative comparisons; 2 folds is enough
BOOTSTRAP       = 400    # resamples for CIs and paired tests

# If the real dataset is absent, fall back to schema-identical synthetic data
# so the notebook still runs end to end (CI, laptop, reviewer).
if not glob.glob(os.path.join(DATA_ROOT, "*_merged_full.csv")):
    DATA_ROOT = os.path.abspath("./synthetic")
    if not glob.glob(os.path.join(DATA_ROOT, "*_merged_full.csv")):
        subprocess.run([sys.executable,
                        os.path.join(PKG_ROOT, "scripts", "make_synthetic.py"),
                        "--out", DATA_ROOT, "--patients", "160",
                        "--genes", "300"], check=True)
    print("!! real dataset not found - running on SYNTHETIC data:", DATA_ROOT)

GPU = torch.cuda.is_available()
cfg = Config()
cfg.data.root              = DATA_ROOT
cfg.data.cohorts           = ["BLCA", "LIHC", "SKCM", "THCA"]
cfg.data.class_target      = "stage"     # stage | grade | cohort | subtype
cfg.data.survival_endpoint = "OS"        # OS | DSS | PFS | DFS
cfg.data.max_genes         = 2000 if GPU else 200
cfg.data.n_genes_per_block = cfg.data.max_genes // 2

# Optional biological priors -- supply these for sharper interpretation.
# cfg.data.ppi_edgelist  = "/kaggle/input/string-ppi/string_human_links.csv"
# cfg.data.pathway_gmt   = "/kaggle/input/msigdb/c2.cp.reactome.v2023.1.symbols.gmt"

cfg.train.device  = "cuda" if GPU else "cpu"
cfg.train.epochs  = 200 if GPU else 30
cfg.train.out_dir = OUT_DIR
cfg.train.batch_size, cfg.train.full_batch = 32, False
cfg.train.outer_folds, cfg.train.n_repeats = OUTER_FOLDS, REPEATS
cfg.tag = "main"

os.makedirs(OUT_DIR, exist_ok=True)
CLASS_NAMES = {"stage": ["I", "II", "III", "IV"], "grade": ["low", "high"]} \
    .get(cfg.data.class_target)
print(json.dumps({"root": cfg.data.root, "target": cfg.data.class_target,
                  "endpoint": cfg.data.survival_endpoint,
                  "max_genes": cfg.data.max_genes, "device": cfg.train.device,
                  "epochs": cfg.train.epochs, "folds": OUTER_FOLDS,
                  "repeats": REPEATS, "RUN_FULL": RUN_FULL}, indent=2))

In [ ]:
# ---- table helpers used throughout --------------------------------------
def fold_table(rows, metrics=None, sort_by=None, ascending=False):
    # {model: [per-fold metric dicts]} -> one row per model, mean +/- sd
    out = {}
    for name, folds in rows.items():
        keys = metrics or sorted({k for f in folds for k in f})
        agg = {}
        for k in keys:
            v = np.array([f.get(k, np.nan) for f in folds], dtype=float)
            v = v[np.isfinite(v)]
            agg[k] = np.nan if v.size == 0 else v.mean()
            agg[k + "_sd"] = 0.0 if v.size < 2 else v.std(ddof=1)
        out[name] = agg
    df = pd.DataFrame(out).T
    if sort_by and sort_by in df:
        df = df.sort_values(sort_by, ascending=ascending)
    return df


def pretty(df, metrics, decimals=3):
    # mean +/- sd strings for the metrics that go in the paper
    show = pd.DataFrame(index=df.index)
    for m in metrics:
        if m in df:
            show[m] = [f"{a:.{decimals}f} ± {b:.{decimals}f}"
                       if np.isfinite(a) else "—"
                       for a, b in zip(df[m], df.get(m + "_sd", df[m] * 0))]
    return show


def save_table(df, name, index_label="model"):
    path = os.path.join(OUT_DIR, f"{name}.csv")
    df.to_csv(path, index_label=index_label)
    try:
        with open(os.path.join(OUT_DIR, f"{name}.md"), "w") as fh:
            fh.write(df.to_markdown())
    except Exception:
        pass
    print("saved", path)
    return df


CLS_METRICS  = ["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted",
                "precision_macro", "recall_macro", "specificity_macro",
                "npv_macro", "auroc", "auprc", "mcc", "kappa",
                "kappa_quadratic", "top2_accuracy", "log_loss", "brier", "ece"]
SURV_METRICS = ["c_index", "c_index_lo", "c_index_hi", "auc@12m", "auc@36m",
                "auc@60m", "ibs", "brier@12m", "brier@36m", "brier@60m",
                "hr_high_vs_low", "hr_per_sd", "logrank_chi2", "logrank_p"]
HEADLINE     = ["balanced_accuracy", "auroc", "f1_macro", "mcc", "c_index",
                "auc@36m", "ibs", "hr_high_vs_low"]
print(f"{len(CLS_METRICS)} classification + {len(SURV_METRICS)} survival metrics "
      "per model")

## 3. Cohorts — Table 1

The loader scans headers first and only materialises the columns it needs, so a
100 000-column merged table never has to fit in memory twice. Omic blocks are
detected by prefix (`MUT_`, `CNA_`, `RNA_`/`EXP_`, `METH_`); a cohort missing a
modality gets a zero-filled channel plus an availability mask, so the encoder
can tell "absent" from "zero".

In [ ]:
exp = Experiment(cfg)
exp.prepare()
sk = exp.skeleton

rows = []
for name, (a, b) in exp.offsets.items():
    coh = exp.cohorts[name]
    ev = np.nan_to_num(coh.event, nan=0)
    lab = coh.label
    rows.append({
        "cohort": name,
        "patients": len(coh),
        "omic_blocks": ", ".join(sorted(coh.blocks)),
        "genes_measured": ", ".join(f"{k}:{len(v.genes)}"
                                    for k, v in sorted(coh.blocks.items())),
        "events": int(ev.sum()),
        "event_rate": round(float(ev.mean()), 3),
        "median_followup_m": round(float(np.nanmedian(coh.time)), 1),
        "max_followup_m": round(float(np.nanmax(coh.time)), 1),
        "labelled": int((lab >= 0).sum()),
        "class_counts": dict(zip(*[a.tolist() for a in
                                   np.unique(lab[lab >= 0], return_counts=True)])),
    })
cohort_table = save_table(pd.DataFrame(rows).set_index("cohort"),
                          "table1_cohorts", index_label="cohort")
display(cohort_table)
print("pooled class distribution:",
      dict(zip(*[a.tolist() for a in
                 np.unique(sk.label[sk.label >= 0], return_counts=True)])))

In [ ]:
from mosaic.metrics import km_curve

fig, ax = plt.subplots(1, 4, figsize=(19, 3.8))

lab = pd.DataFrame({"cohort": [sk.cohort_names[c] for c in sk.cohort_id],
                    "label": sk.label})
lab = lab[lab.label >= 0]
ct = pd.crosstab(lab.cohort, lab.label)
if CLASS_NAMES:
    ct.columns = [CLASS_NAMES[c] if c < len(CLASS_NAMES) else c for c in ct.columns]
ct.plot(kind="bar", stacked=True, ax=ax[0], colormap="viridis")
ax[0].set_title(f"{cfg.data.class_target} by cohort"); ax[0].set_xlabel("")

ax[1].hist(sk.time[np.isfinite(sk.time)], bins=40, color="#4C72B0")
ax[1].set_title(f"{cfg.data.survival_endpoint} follow-up"); ax[1].set_xlabel("months")

for ci, name in enumerate(sk.cohort_names):
    m = sk.cohort_id == ci
    t, s = km_curve(sk.time[m], np.nan_to_num(sk.event[m], nan=0))
    ax[2].step(t, s, where="post", label=name)
ax[2].set_title("Kaplan-Meier by cohort"); ax[2].set_xlabel("months")
ax[2].set_ylim(0, 1.02); ax[2].legend(fontsize=8)

ev = pd.DataFrame({"cohort": [sk.cohort_names[c] for c in sk.cohort_id],
                   "event": np.nan_to_num(sk.event, nan=0)})
ev.groupby("cohort")["event"].mean().plot(kind="bar", ax=ax[3], color="#C44E52")
ax[3].set_title("Event rate"); ax[3].set_xlabel(""); ax[3].set_ylim(0, 1)

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_cohorts.png", dpi=200,
                                bbox_inches="tight"); plt.show()

## 4. One fold, end to end — Table 2

`build_fold` re-fits **everything** that could leak — gene selection, the gene
graph, scalers, survival bin edges — on the training rows only. Held-out
patients are passed through `transform` and attach to training anchors in the
patient graph, never to each other.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from mosaic.data.preprocess import stratification_key

strat = stratification_key(sk)
idx = np.arange(len(sk))
skf = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True,
                      random_state=cfg.train.seed)
SPLITS = list(skf.split(idx, strat))
train_idx, test_idx = SPLITS[0]

inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=cfg.train.seed)
fit_i, val_i = next(inner.split(train_idx, strat[train_idx]))
fit_idx, val_idx = train_idx[fit_i], train_idx[val_i]
n_classes = max(int(sk.label.max()) + 1, 2)
print(f"train={len(fit_idx)}  val={len(val_idx)}  test={len(test_idx)}  "
      f"classes={n_classes}")

tensors, prep, data = exp.build_fold(train_idx)
BIN_EDGES = np.concatenate([[0.0], prep.surv_bins])
print(tensors.graph.summary())
print("node features:", tuple(tensors.x.shape), "| clinical:",
      tuple(tensors.clinical.shape))

In [ ]:
t0 = time.time()
model = exp.train_fold(tensors, fit_idx, val_idx, n_classes)
print(f"trained in {time.time() - t0:.0f}s")

fold_result = exp.evaluate_fold(model, tensors, test_idx, train_idx,
                                bin_edges=BIN_EDGES)
scalar = {k: v for k, v in fold_result.items()
          if isinstance(v, (int, float)) and not k.startswith("_")}
table2 = pd.DataFrame({"MOSAIC-GNN (fold 1)": scalar}).round(4)
save_table(table2, "table2_single_fold", index_label="metric")
display(table2)

In [ ]:
from mosaic.metrics import per_class_report

pred = exp.predict(model, tensors, test_idx, anchor_idx=train_idx,
                   mc_samples=cfg.train.mc_dropout_samples)
y_te = tensors.label.numpy()[test_idx]
t_te = tensors.time.numpy()[test_idx]
e_te = tensors.event.numpy()[test_idx]

print("Per-class report (Table 2b)")
table2b = save_table(per_class_report(y_te, pred["prob"], CLASS_NAMES).round(3),
                     "table2b_per_class", index_label="row")
display(table2b)

print("Per-cohort breakdown (Table 2c)")
table2c = save_table(pd.DataFrame(fold_result["per_cohort"]).T.round(3),
                     "table2c_per_cohort", index_label="cohort")
display(table2c)

### 4.1 Figure 2 — risk stratification

In [ ]:
from mosaic.metrics import logrank_test, univariate_cox

cuts = np.quantile(pred["risk"], [1/3, 2/3])
group = np.digitize(pred["risk"], cuts)
gnames = ["low risk", "intermediate", "high risk"]
colors = ["#2E7D32", "#F9A825", "#C62828"]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for g in range(3):
    m = group == g
    if m.sum() < 5:
        continue
    tt, ss = km_curve(t_te[m], e_te[m])
    ax[0].step(tt, ss, where="post", color=colors[g],
               label=f"{gnames[g]} (n={int(m.sum())})")
hl = np.isin(group, [0, 2])
chi2, p = logrank_test(t_te[hl], e_te[hl], (group[hl] == 2).astype(int))
hr = univariate_cox(t_te[hl], e_te[hl], (group[hl] == 2).astype(float))
ax[0].set_title(f"Risk tertiles — log-rank p={p:.2e}\n"
                f"HR (high vs low) = {hr['hr']:.2f} "
                f"[{hr['hr_low']:.2f}, {hr['hr_high']:.2f}]")
ax[0].set_xlabel("months"); ax[0].set_ylabel("survival probability")
ax[0].set_ylim(0, 1.02); ax[0].legend(fontsize=8)

for g, c in zip(range(3), colors):
    m = group == g
    if m.any():
        ax[1].step(BIN_EDGES[:pred["survival"].shape[1]],
                   pred["survival"][m].mean(0), where="post", color=c,
                   label=gnames[g])
ax[1].set_title("Predicted survival curves (hazard head)")
ax[1].set_xlabel("months"); ax[1].set_ylim(0, 1.02); ax[1].legend(fontsize=8)

# observed vs predicted survival at a fixed horizon = calibration
h = 36.0
col = int(np.clip(np.searchsorted(BIN_EDGES, h, side="right") - 1, 0,
                  pred["survival"].shape[1] - 1))
predicted, observed = [], []
qs = np.quantile(pred["survival"][:, col], np.linspace(0, 1, 6))
for lo, hi in zip(qs[:-1], qs[1:]):
    m = (pred["survival"][:, col] >= lo) & (pred["survival"][:, col] <= hi)
    if m.sum() < 5:
        continue
    tt, ss = km_curve(t_te[m], e_te[m])
    observed.append(float(ss[np.searchsorted(tt, h, side="right") - 1]))
    predicted.append(float(pred["survival"][m, col].mean()))
ax[2].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[2].scatter(predicted, observed, s=60, color="#4C72B0")
ax[2].set_xlabel(f"predicted S({int(h)}m)"); ax[2].set_ylabel("observed (KM)")
ax[2].set_title("Survival calibration"); ax[2].set_xlim(0, 1); ax[2].set_ylim(0, 1)

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_risk.png", dpi=200,
                                bbox_inches="tight"); plt.show()

### 4.2 Figure 3 — classification quality, calibration, uncertainty

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve

fig, ax = plt.subplots(1, 4, figsize=(19, 4.2))
m = y_te >= 0

ConfusionMatrixDisplay.from_predictions(
    y_te[m], pred["prob"][m].argmax(1), normalize="true", cmap="Blues",
    colorbar=False, ax=ax[0],
    display_labels=CLASS_NAMES[:n_classes] if CLASS_NAMES else None)
ax[0].set_title(f"{cfg.data.class_target} (row-normalised)")

for c in np.unique(y_te[m]):
    fpr, tpr, _ = roc_curve((y_te[m] == c).astype(int), pred["prob"][m][:, c])
    lbl = CLASS_NAMES[c] if CLASS_NAMES and c < len(CLASS_NAMES) else c
    ax[1].plot(fpr, tpr, label=f"{lbl} (AUC={auc(fpr, tpr):.2f})")
ax[1].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR")
ax[1].set_title("One-vs-rest ROC"); ax[1].legend(fontsize=8)

# reliability diagram
conf = pred["prob"][m].max(1); correct = (pred["prob"][m].argmax(1) == y_te[m])
edges = np.linspace(0, 1, 11)
xs, ys = [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    b = (conf > lo) & (conf <= hi)
    if b.sum() >= 5:
        xs.append(conf[b].mean()); ys.append(correct[b].mean())
ax[2].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[2].plot(xs, ys, "o-", color="#DD8452")
ax[2].set_xlabel("confidence"); ax[2].set_ylabel("accuracy")
ax[2].set_title(f"Reliability (ECE={scalar.get('ece', float('nan')):.3f})")

order = np.argsort(pred["entropy"][m])
corr = correct[order]
cov = np.arange(1, len(order) + 1) / len(order)
ax[3].plot(cov, np.cumsum(corr) / np.arange(1, len(order) + 1))
ax[3].axhline(corr.mean(), ls="--", c="grey", lw=0.8, label="full coverage")
ax[3].set_xlabel("coverage (most confident first)"); ax[3].set_ylabel("accuracy")
ax[3].set_title("Selective prediction (MC dropout)"); ax[3].legend(fontsize=8)

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_classification.png", dpi=200,
                                bbox_inches="tight"); plt.show()

## 5. Overfitting diagnostics — Tables S1-S3

With ~10^4 molecular features and a few hundred patients, "does it generalise"
is the first question a reviewer asks. Four checks answer it, and all four
belong in the supplement:

* **generalisation gap** — the same metrics on train / validation / test;
* **learning curves** — where train and validation separate, i.e. whether early
  stopping fired in the right place;
* **label permutation test** — retrain on shuffled labels; if anything leaks,
  performance stays above chance. This is the only check that can *prove* the
  absence of leakage, and it is the one most papers skip;
* **sample-size curve** — is the model data-limited or capacity-limited?

Regularisation in force: gene dropout 0.15 (whole genes are hidden each step,
value and mask channel together), input jitter, dropout 0.4, edge dropout 0.2,
weight decay 1e-3, label smoothing 0.1, gradient clipping, stochastic weight
averaging over the last 40 % of training, and early stopping on a validation
criterion that weights both tasks.

In [ ]:
from mosaic.diagnostics import (generalisation_gap, learning_curves,
                                parameter_report, permutation_p_value,
                                permutation_test, sample_size_curve)

cap = parameter_report(model, len(fit_idx))
print(json.dumps({k: (round(v, 2) if isinstance(v, float) else v)
                  for k, v in cap.items()}, indent=2))

tableS1 = generalisation_gap(
    exp, tensors, model,
    {"train": fit_idx, "validation": val_idx, "test": test_idx}, train_idx)
save_table(tableS1[["balanced_accuracy", "auroc", "f1_macro", "mcc",
                    "log_loss", "brier", "c_index", "n"]].round(4),
           "tableS1_generalisation_gap", index_label="split")
display(tableS1[["balanced_accuracy", "auroc", "f1_macro", "mcc", "log_loss",
                 "brier", "c_index", "n"]].round(3))

In [ ]:
curves = learning_curves(model)
if not curves.empty:
    save_table(curves.round(4), "tableS1b_learning_curve", index_label="epoch")
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(curves.index, curves["train_balanced_accuracy"], "o-",
               label="train")
    ax[0].plot(curves.index, curves["val_balanced_accuracy"], "s-",
               label="validation")
    ax[0].set_title("Balanced accuracy"); ax[0].set_xlabel("epoch")
    ax[0].legend(fontsize=8)

    ax[1].plot(curves.index, curves["train_c_index"], "o-", label="train")
    ax[1].plot(curves.index, curves["val_c_index"], "s-", label="validation")
    ax[1].axhline(0.5, color="grey", ls="--", lw=0.8)
    ax[1].set_title("C-index"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=8)

    gap = curves["train_score"] - curves["val_score"]
    ax[2].plot(curves.index, gap, "o-", color="#C44E52")
    ax[2].axhline(0, color="k", lw=0.8)
    ax[2].set_title("Generalisation gap (train - validation)")
    ax[2].set_xlabel("epoch")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figS1_learning_curves.png",
                                    dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# The smallest attainable p-value is 1/(N_PERM + 1): 20 permutations can just
# reach p = 0.048, which is the bare minimum for a claim at alpha = 0.05.
# Use >= 100 for the manuscript (it is embarrassingly parallel across seeds).
N_PERM = 20 if RUN_FULL else 2
null = permutation_test(exp, tensors, fit_idx, val_idx, test_idx, n_classes,
                        n_permutations=N_PERM, seed=cfg.train.seed)
obs = {"balanced_accuracy": tableS1.loc["test", "balanced_accuracy"],
       "auroc": tableS1.loc["test", "auroc"],
       "c_index": tableS1.loc["test", "c_index"]}
tableS2 = pd.DataFrame({
    "observed": obs,
    "null_mean": null.mean(),
    "null_max": null.max(),
    "chance": {"balanced_accuracy": 1.0 / n_classes, "auroc": 0.5,
               "c_index": 0.5},
    "p_permutation": {k: permutation_p_value(obs[k], null[k]) for k in obs},
})
save_table(tableS2.round(4), "tableS2_permutation_test", index_label="metric")
display(tableS2.round(3))
print(f"n_permutations = {N_PERM}; the smallest attainable p is "
      f"{1 / (N_PERM + 1):.3f}")

In [ ]:
tableS3 = sample_size_curve(exp, tensors, fit_idx, val_idx, test_idx, n_classes,
                            fractions=(0.25, 0.5, 0.75, 1.0),
                            seed=cfg.train.seed)
save_table(tableS3.round(4), "tableS3_sample_size", index_label="n_train")
display(tableS3.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tableS3.index, tableS3["train_balanced_accuracy"], "o-", label="train")
ax[0].plot(tableS3.index, tableS3["test_balanced_accuracy"], "s-", label="test")
ax[0].axhline(1.0 / n_classes, color="grey", ls="--", lw=0.8, label="chance")
ax[0].set_xlabel("training patients"); ax[0].set_ylabel("balanced accuracy")
ax[0].set_title("Learning curve vs sample size"); ax[0].legend(fontsize=8)

ax[1].plot(tableS3.index, tableS3["train_c_index"], "o-", label="train")
ax[1].plot(tableS3.index, tableS3["test_c_index"], "s-", label="test")
ax[1].axhline(0.5, color="grey", ls="--", lw=0.8)
ax[1].set_xlabel("training patients"); ax[1].set_ylabel("C-index")
ax[1].set_title("Survival head vs sample size"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/figS2_sample_size.png", dpi=200,
                                bbox_inches="tight"); plt.show()

## 6. Head-to-head cross-validation — Table 3

MOSAIC-GNN and **every baseline are trained on identical folds**, so the
comparison is paired and a paired bootstrap test is valid. Four families of
comparator are included:

* classical predictors on concatenated features (penalised regression, SVMs,
  tree ensembles, gradient boosting, kNN, LDA, naive Bayes, PCA+logistic);
* multi-omics fusion networks (early fusion MLP, late fusion, MOGONET-style
  per-omic GCN with view fusion);
* patient-graph GNNs (GCN, GAT, GraphSAGE on a fixed kNN patient graph) — what
  most published multi-omics GNNs actually are;
* dedicated survival models (Cox-lasso, Cox-ridge, clinical-only Cox, random
  survival forest, DeepSurv).

In [ ]:
from mosaic.baselines import run_baseline_suite

per_model_folds, pooled = {}, {}
BLOCKS = sorted(sk.blocks)

if RUN_FULL:
    for rep in range(REPEATS):
        splits = SPLITS if rep == 0 else list(StratifiedKFold(
            n_splits=OUTER_FOLDS, shuffle=True,
            random_state=cfg.train.seed + rep).split(idx, strat))
        for f, (tr, te) in enumerate(splits):
            print(f"\n===== repeat {rep+1}/{REPEATS} fold {f+1}/{OUTER_FOLDS} =====")
            tn, pp, dt = exp.build_fold(tr)
            edges = np.concatenate([[0.0], pp.surv_bins])

            i_fit, i_val = next(inner.split(tr, strat[tr]))
            mdl = exp.train_fold(tn, tr[i_fit], tr[i_val], n_classes, verbose=False)
            res = exp.evaluate_fold(mdl, tn, te, tr, bin_edges=edges)
            mp = exp.predict(mdl, tn, te, anchor_idx=tr)
            per_model_folds.setdefault("MOSAIC-GNN", []).append(
                {k: v for k, v in res.items() if isinstance(v, (int, float))})
            pooled.setdefault("MOSAIC-GNN", {"prob": [], "risk": [], "y": [],
                                             "t": [], "e": []})
            pooled["MOSAIC-GNN"]["prob"].append(mp["prob"])
            pooled["MOSAIC-GNN"]["risk"].append(mp["risk"])
            print(f"  {'MOSAIC-GNN':<24} bacc={res.get('balanced_accuracy', np.nan):.3f} "
                  f"auroc={res.get('auroc', np.nan):.3f} "
                  f"c-index={res.get('c_index', np.nan):.3f}")

            mets, preds = run_baseline_suite(
                tn.x.numpy(), tn.clinical.numpy(), dt.label, dt.time, dt.event,
                tr, te, n_classes, block_names=BLOCKS, seed=cfg.train.seed,
                device=str(exp.device), epochs=200 if GPU else 150)
            for name, mm in mets.items():
                per_model_folds.setdefault(name, []).append(mm)
                p = pooled.setdefault(name, {"prob": [], "risk": []})
                p["prob"].append(preds[name]["prob"])
                p["risk"].append(preds[name]["risk"])
            # truth arrays, aligned with the pooled predictions
            pooled.setdefault("_truth", {"y": [], "t": [], "e": []})
            pooled["_truth"]["y"].append(dt.label[te])
            pooled["_truth"]["t"].append(dt.time[te])
            pooled["_truth"]["e"].append(np.nan_to_num(dt.event[te], nan=0.0))
    print("\ndone")
else:
    print("RUN_FULL is False - skipping the head-to-head comparison.")

In [ ]:
table3 = table3_full = None
if per_model_folds:
    table3_full = fold_table(per_model_folds, sort_by="c_index")
    save_table(table3_full.round(4), "table3_comparison_full")
    table3 = pretty(table3_full, HEADLINE)
    save_table(table3, "table3_comparison_headline")
    display(table3)

In [ ]:
# All classification metrics, all models.
if table3_full is not None:
    display(pretty(table3_full.sort_values("balanced_accuracy", ascending=False),
                   CLS_METRICS))

In [ ]:
# All survival metrics, all models.
if table3_full is not None:
    display(pretty(table3_full.sort_values("c_index", ascending=False),
                   SURV_METRICS))

### 6.1 Table 4 — paired bootstrap tests against MOSAIC-GNN

Predictions are pooled across folds and resampled jointly, so each test asks:
*on the same patients, is MOSAIC-GNN better than this baseline?*

In [ ]:
from mosaic.metrics import (balanced_accuracy_score, concordance_index,
                            paired_bootstrap_test)

table4 = None
if pooled and "_truth" in pooled:
    y_all = np.concatenate(pooled["_truth"]["y"])
    t_all = np.concatenate(pooled["_truth"]["t"])
    e_all = np.concatenate(pooled["_truth"]["e"])
    ref = pooled["MOSAIC-GNN"]
    ref_prob = np.concatenate(ref["prob"]); ref_risk = np.concatenate(ref["risk"])

    rows = []
    for name, p in pooled.items():
        if name in ("_truth", "MOSAIC-GNN"):
            continue
        row = {"model": name}
        probs = [q for q in p["prob"] if q is not None]
        if len(probs) == len(p["prob"]):
            pb = np.concatenate(probs)
            m = y_all >= 0
            r = paired_bootstrap_test(
                lambda yy, pp_: balanced_accuracy_score(yy, pp_.argmax(1)),
                y_all[m], ref_prob[m], pb[m], n_boot=BOOTSTRAP,
                seed=cfg.train.seed)
            row.update({"d_bacc": r["delta"], "p_bacc": r["p_value"]})
        risks = [q for q in p["risk"] if q is not None]
        if len(risks) == len(p["risk"]):
            rk = np.concatenate(risks)
            stacked = np.stack([t_all, e_all], 1)
            r = paired_bootstrap_test(
                lambda te_, rr: concordance_index(te_[:, 0], rr, te_[:, 1]),
                stacked, ref_risk, rk, n_boot=BOOTSTRAP, seed=cfg.train.seed)
            row.update({"d_c_index": r["delta"], "p_c_index": r["p_value"]})
        rows.append(row)

    table4 = pd.DataFrame(rows).set_index("model")
    # Benjamini-Hochberg across the comparison family.
    for col in ("p_bacc", "p_c_index"):
        if col in table4:
            pv = table4[col].values.astype(float)
            ok = np.isfinite(pv)
            order = np.argsort(np.where(ok, pv, np.inf))
            adj = np.full_like(pv, np.nan)
            n_ok = ok.sum()
            run_min = 1.0
            for rank, i in enumerate(order[:n_ok][::-1]):
                run_min = min(run_min, pv[i] * n_ok / (n_ok - rank))
                adj[i] = run_min
            table4[col.replace("p_", "q_")] = adj
    save_table(table4.round(4), "table4_paired_tests")
    display(table4.round(4))
    print("positive delta = MOSAIC-GNN better; q = Benjamini-Hochberg adjusted")

### 6.2 Figure 4 — model comparison

In [ ]:
if table3_full is not None:
    fig, ax = plt.subplots(1, 3, figsize=(19, max(5, 0.32 * len(table3_full))))

    d = table3_full.dropna(subset=["c_index"]).sort_values("c_index")
    err = np.nan_to_num(np.abs(np.vstack([d["c_index"] - d["c_index_lo"],
                                          d["c_index_hi"] - d["c_index"]])))
    cols = ["#C62828" if i == "MOSAIC-GNN" else "#4C72B0" for i in d.index]
    ax[0].errorbar(d["c_index"], range(len(d)), xerr=err, fmt="o", ecolor="grey",
                   elinewidth=1, capsize=2, ls="none")
    ax[0].scatter(d["c_index"], range(len(d)), color=cols, zorder=3)
    ax[0].set_yticks(range(len(d))); ax[0].set_yticklabels(d.index, fontsize=8)
    ax[0].axvline(0.5, color="grey", ls="--", lw=0.8)
    ax[0].set_xlabel("C-index (95% bootstrap CI)"); ax[0].set_title("Discrimination")

    d2 = table3_full.dropna(subset=["balanced_accuracy"]).sort_values(
        "balanced_accuracy")
    cols = ["#C62828" if i == "MOSAIC-GNN" else "#55A868" for i in d2.index]
    ax[1].barh(range(len(d2)), d2["balanced_accuracy"],
               xerr=d2["balanced_accuracy_sd"], color=cols, capsize=2)
    ax[1].set_yticks(range(len(d2))); ax[1].set_yticklabels(d2.index, fontsize=8)
    ax[1].axvline(1.0 / n_classes, color="grey", ls="--", lw=0.8, label="chance")
    ax[1].set_xlabel("balanced accuracy"); ax[1].set_title("Classification")
    ax[1].legend(fontsize=8)

    d3 = table3_full.dropna(subset=["auroc"]).sort_values("auroc")
    cols = ["#C62828" if i == "MOSAIC-GNN" else "#8172B3" for i in d3.index]
    ax[2].barh(range(len(d3)), d3["auroc"], xerr=d3["auroc_sd"], color=cols,
               capsize=2)
    ax[2].set_yticks(range(len(d3))); ax[2].set_yticklabels(d3.index, fontsize=8)
    ax[2].axvline(0.5, color="grey", ls="--", lw=0.8)
    ax[2].set_xlabel("AUROC (macro OvR)"); ax[2].set_title("Ranking quality")

    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig4_comparison.png", dpi=200,
                                    bbox_inches="tight"); plt.show()

## 7. Ablations — Table 5

Each row removes exactly one component. A component earns its place in the
architecture only if removing it costs performance.

In [ ]:
from mosaic.cli import ABLATIONS
from mosaic.config import Config as C

ablation_folds, table5 = {}, None
if RUN_FULL:
    for name, override in ABLATIONS.items():
        mcfg = type(cfg.model)(**{**cfg.model.__dict__, **override})
        sub = C(data=cfg.data, graph=cfg.graph, model=mcfg, loss=cfg.loss,
                train=type(cfg.train)(**cfg.train.__dict__), tag=f"abl-{name}")
        e = Experiment(sub); e.prepare()
        for f, (tr, te) in enumerate(SPLITS[:ABLATION_FOLDS]):
            tn, pp, _ = e.build_fold(tr)
            i_fit, i_val = next(inner.split(tr, strat[tr]))
            mdl = e.train_fold(tn, tr[i_fit], tr[i_val], n_classes, verbose=False)
            r = e.evaluate_fold(mdl, tn, te, tr,
                                bin_edges=np.concatenate([[0.0], pp.surv_bins]))
            ablation_folds.setdefault(name, []).append(
                {k: v for k, v in r.items() if isinstance(v, (int, float))})
        m = ablation_folds[name]
        print(f"{name:<18} bacc={np.nanmean([x.get('balanced_accuracy', np.nan) for x in m]):.3f} "
              f"c-index={np.nanmean([x.get('c_index', np.nan) for x in m]):.3f}")

    table5 = fold_table(ablation_folds)
    save_table(table5.round(4), "table5_ablation", index_label="variant")
    display(pretty(table5, HEADLINE))
else:
    print("RUN_FULL is False - skipping ablations.")

In [ ]:
if table5 is not None and "full" in table5.index:
    delta = table5[HEADLINE].drop(index="full") - table5.loc["full", HEADLINE]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    delta.plot(kind="barh", ax=ax, width=0.8, colormap="tab10")
    ax.axvline(0, color="k", lw=1)
    ax.set_title("Ablation: change vs the full model\n"
                 "(negative = the removed component was helping)")
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig5_ablation.png", dpi=200,
                                    bbox_inches="tight"); plt.show()
    display(delta.round(4))

## 8. Leave-one-cancer-out transfer — Table 6

Train on three cohorts, test on the fourth, which the model has never seen.
This is the experiment that separates a genuine pan-cancer representation from
a cohort detector, and it is where the cohort-adversarial latent split has to
pay for itself.

In [ ]:
loco_folds, table6 = {}, None
if RUN_FULL:
    for held in cfg.data.cohorts:
        a, b = exp.offsets[held]
        te = np.arange(a, b)
        tr = np.setdiff1d(idx, te)
        rng = np.random.default_rng(cfg.train.seed)
        va = rng.choice(tr, size=max(int(0.15 * len(tr)), 20), replace=False)
        ft = np.setdiff1d(tr, va)
        tn, pp, _ = exp.build_fold(tr)
        mdl = exp.train_fold(tn, ft, va, n_classes, verbose=False)
        r = exp.evaluate_fold(mdl, tn, te, tr,
                              bin_edges=np.concatenate([[0.0], pp.surv_bins]))
        loco_folds[held] = [{k: v for k, v in r.items()
                             if isinstance(v, (int, float))}]
        print(f"{held:<6} n={len(te):4d} bacc={r.get('balanced_accuracy', np.nan):.3f} "
              f"auroc={r.get('auroc', np.nan):.3f} c-index={r.get('c_index', np.nan):.3f} "
              f"logrank_p={r.get('logrank_p', np.nan):.2e}")

    table6 = fold_table(loco_folds)
    save_table(table6.round(4), "table6_loco", index_label="held_out_cohort")
    display(table6[HEADLINE].round(3))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    table6[["balanced_accuracy", "auroc", "c_index"]].plot(
        kind="bar", ax=ax[0], colormap="Set2")
    ax[0].axhline(0.5, color="grey", ls="--", lw=0.8)
    ax[0].set_title("Leave-one-cancer-out performance"); ax[0].set_xlabel("held-out cohort")
    if table3_full is not None and "MOSAIC-GNN" in table3_full.index:
        ref = table3_full.loc["MOSAIC-GNN", ["balanced_accuracy", "auroc", "c_index"]]
        (table6[["balanced_accuracy", "auroc", "c_index"]] - ref).plot(
            kind="bar", ax=ax[1], colormap="Set1")
        ax[1].axhline(0, color="k", lw=1)
        ax[1].set_title("Transfer gap vs pooled cross-validation")
        ax[1].set_xlabel("held-out cohort")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig6_loco.png", dpi=200,
                                    bbox_inches="tight"); plt.show()
else:
    print("RUN_FULL is False - skipping leave-one-cancer-out.")

## 9. Interpretation — Tables 7 & 8

Three complementary attributions, because attention on its own is not evidence:
module attention (which pathways drive the embedding), the within-module gene
gate (which genes dominate), and integrated gradients on the node features
(signed, per-omic, baseline-referenced).

In [ ]:
from mosaic.explain import (collect_attention, export_ranked_gene_list,
                            gene_importance_table, integrated_gradients,
                            module_importance_table, personalised_subnetwork)

att = collect_attention(model, tensors, test_idx, exp.device)
n_ig, ig_steps = (128, 32) if GPU else (48, 8)
ig = integrated_gradients(model, tensors, test_idx[:min(len(test_idx), n_ig)],
                          exp.device, target="risk", steps=ig_steps)

table7 = gene_importance_table(tensors.graph.genes, tensors.graph.modules,
                               tensors.graph.module_names, att["gene_gate"],
                               ig, BLOCKS)
table8 = module_importance_table(tensors.graph.module_names,
                                 att["module_attention"], tensors.graph.modules,
                                 tensors.graph.genes, gate=att["gene_gate"])
save_table(table7.round(6), "table7_gene_importance", index_label="rank")
save_table(table8.round(6), "table8_module_importance", index_label="rank")
export_ranked_gene_list(table7, f"{OUT_DIR}/genes.rnk")   # GSEA / g:Profiler
display(table7.head(25))
display(table8.head(12))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5))

top = table7.head(25).iloc[::-1]
score_col = "ig_abs" if "ig_abs" in top else "gate_mean"
ax[0].barh(top["gene"], top[score_col], color="#4C72B0")
ax[0].set_title(f"Top genes by {score_col}"); ax[0].tick_params(labelsize=8)

ig_cols = [c for c in table7.columns if c.startswith("ig_") and c != "ig_abs"]
if ig_cols:
    tt = table7.head(20).set_index("gene")[ig_cols].iloc[::-1]
    tt.plot(kind="barh", stacked=True, ax=ax[1], colormap="coolwarm")
    ax[1].axvline(0, color="k", lw=0.8)
    ax[1].set_title("Signed attribution by omic layer")
    ax[1].tick_params(labelsize=8); ax[1].legend(fontsize=8)

coh_te = tensors.cohort.numpy()[test_idx]
present = [c for c in range(len(sk.cohort_names)) if (coh_te == c).sum() > 0]
mat = np.vstack([att["module_attention"][coh_te == c].mean(0) for c in present])
keep = np.argsort(-mat.mean(0))[:30]
im = ax[2].imshow(mat[:, keep], aspect="auto", cmap="magma")
ax[2].set_yticks(range(len(present)))
ax[2].set_yticklabels([sk.cohort_names[c] for c in present])
ax[2].set_xticks(range(len(keep)))
ax[2].set_xticklabels([tensors.graph.module_names[i] for i in keep],
                      rotation=90, fontsize=6)
ax[2].set_title("Module attention by cohort")
plt.colorbar(im, ax=ax[2], fraction=0.03)

plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig7_interpretation.png", dpi=200,
                                bbox_inches="tight"); plt.show()

In [ ]:
# One patient's rewired interaction network -- edge weights are that patient's
# own attention over the shared topology.
sub = personalised_subnetwork(model, tensors, int(test_idx[0]), exp.device,
                              top_edges=150)
save_table(sub.round(5), "table9_patient_subnetwork", index_label="edge")
display(sub.head(15))
print("export to Cytoscape / networkx for the subnetwork figure")

## 10. Everything that was produced

In [ ]:
produced = sorted(glob.glob(os.path.join(OUT_DIR, "*")))
print(f"{len(produced)} files in {OUT_DIR}\n")
for f in produced:
    print(f"  {os.path.basename(f):<40} {os.path.getsize(f) / 1024:8.1f} KB")

## 11. Before you write it up

- Report **mean ± sd over ≥ 3 repeats × 5 folds** (`REPEATS = 3`,
  `OUTER_FOLDS = 5`); the settings above are tuned to finish quickly.
- Baselines were tuned on the *same* inner splits — say so explicitly.
- Table 4 gives the **paired bootstrap** p-values with Benjamini-Hochberg
  correction; quote the adjusted `q`, not the raw `p`.
- Show **per-cohort** numbers (Table 2c) next to the pooled figure.
- Report the **leave-one-cancer-out** result even where it is weak — it is the
  honest measure of "pan-cancer".
- Cross-check the top-ranked genes (Table 7) against COSMIC CGC / OncoKB and
  present the novel ones as **hypotheses**, not findings.
- State the limitations: cohort sizes in the hundreds → wide survival CIs; the
  co-alteration fallback graph is not a validated interaction network; and
  attention-derived rankings need orthogonal validation.